## Fetching Cosponsor Names via the congress.gov API


In [1]:
# imports
import os
import re
import time
import requests
import pandas as pd

In [2]:
# api setup
API_KEY  = os.environ.get('CONGRESS_API_KEY', 'OtWuMddeA7AEheFMyj7zPCqtN7YTJB69CjhGYjpy')
BASE_URL = 'https://api.congress.gov/v3'
DELAY    = 0.5  # seconds between requests

In [4]:
# load the cleaned bills dataframe
bills_df = pd.read_csv('../../data/bills/bills_with_topic.csv', index_col=0)

In [5]:
def parse_bill_ref(legislation_number, congress_str):
    """
    Convert 'H.R. 7933' + '119th Congress (2025-2026)' into (119, 'hr', '7933').
    Returns None if parsing fails.
    """
    # extract congress number from string like '119th Congress (2025-2026)'
    m_congress = re.match(r'(\d+)', str(congress_str))
    if not m_congress:
        return None
    congress_num = int(m_congress.group(1))

    # map legislation number prefix to API type string
    type_map = {
        'H.R.':    'hr',
        'S.':      's',
        'H.Res.':  'hres',
        'S.Res.':  'sres',
        'H.Con.Res.': 'hconres',
        'S.Con.Res.': 'sconres',
        'H.J.Res.':   'hjres',
        'S.J.Res.':   'sjres',
    }
    bill_type = None
    bill_number = None
    for prefix, api_type in type_map.items():
        if str(legislation_number).startswith(prefix):
            bill_type   = api_type
            bill_number = str(legislation_number).replace(prefix, '').strip()
            break

    if bill_type is None:
        return None

    return congress_num, bill_type, bill_number


def get_cosponsors(congress_num, bill_type, bill_number, api_key):
    """
    Fetch cosponsors for one bill. Returns a list of bioguide IDs,
    or an empty list if there are none or the request fails.
    """
    url = (
        f'{BASE_URL}/bill/{congress_num}/{bill_type}/{bill_number}/cosponsors'
        f'?api_key={api_key}&format=json&limit=250'
    )
    resp = requests.get(url)
    if resp.status_code == 404:
        return []   # bill exists but has no cosponsor endpoint — treat as zero
    resp.raise_for_status()
    data = resp.json()

    cosponsors = []
    for cs in data.get('cosponsors', []):
        bioguide_id = cs.get('bioguideId')
        if bioguide_id:
            cosponsors.append(bioguide_id)
    return cosponsors

In [7]:
# only bother fetching for bills that actually have cosponsors
bills_to_fetch = bills_df[bills_df['Number of Cosponsors'] > 0].copy()
print(f'{len(bills_to_fetch)} bills with at least one cosponsor')

1866 bills with at least one cosponsor


In [8]:
cosponsor_map = {}   # bill legislation_number -> list of bioguide IDs

for i, row in bills_to_fetch.iterrows():
    ref = parse_bill_ref(row['Legislation Number'], row['Congress'])
    if ref is None:
        continue

    congress_num, bill_type, bill_number = ref

    try:
        cosponsors = get_cosponsors(congress_num, bill_type, bill_number, API_KEY)
        cosponsor_map[row['Legislation Number']] = cosponsors
    except Exception as e:
        print(f'  error on {row["Legislation Number"]}: {e}')
        cosponsor_map[row['Legislation Number']] = []

    if i % 100 == 0:
        print(f'  {i}/{len(bills_to_fetch)} done...')

    time.sleep(DELAY)

print(f'Done. Fetched cosponsor lists for {len(cosponsor_map)} bills.')

  100/1866 done...
  200/1866 done...
  300/1866 done...
  400/1866 done...
  500/1866 done...
  600/1866 done...
  700/1866 done...
  900/1866 done...
  1000/1866 done...
  1300/1866 done...
  1400/1866 done...
  1500/1866 done...
  1600/1866 done...
  1700/1866 done...
  1800/1866 done...
  2100/1866 done...
  2200/1866 done...
  2300/1866 done...
Done. Fetched cosponsor lists for 1866 bills.


In [9]:
# add cosponsor_ids column to bills_df
# bills with 0 cosponsors get an empty list
bills_df['cosponsor_ids'] = bills_df['Legislation Number'].map(
    lambda x: cosponsor_map.get(x, [])
)

bills_df[['Legislation Number', 'Number of Cosponsors', 'cosponsor_ids']].head(10)

,Legislation Number,Number of Cosponsors,cosponsor_ids
0,H.R. 7933,0,[]
1,H.R. 7932,11,"[K000399, T000487, D000230, E000071, H001101, ..."
2,H.R. 7931,1,"[C001136, M001210]"
3,H.R. 7930,6,"[G000586, L000602, N000147, O000173, R000617, ..."
4,H.R. 7929,0,[]
5,H.R. 7928,0,[]
6,H.R. 7927,0,[]
7,H.R. 7926,0,[]
8,H.R. 7925,1,"[B001309, B000825, F000469, H001101]"
9,H.R. 7924,0,[]


In [10]:
bills_df.head()

,Legislation Number,URL,Congress,Title,Sponsor,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors,topic,cosponsor_ids
0,H.R. 7933,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 10, United States Code, to expa...","Watson Coleman, Bonnie [Rep.-D-NJ-12]",2026-03-12,House - Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,0,healthcare,[]
1,H.R. 7932,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),HONOR Gold Star Families Act,"Van Epps, Matt [Rep.-R-TN-7]",2026-03-12,House - Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,11,veterans,"[K000399, T000487, D000230, E000071, H001101, ..."
2,H.R. 7931,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To amend the Employee Retirement Income Securi...,"Van Drew, Jefferson [Rep.-R-NJ-2]",2026-03-12,House - Education and Workforce,Referred to the House Committee on Education a...,2026-03-12,1,public_safety,"[C001136, M001210]"
3,H.R. 7930,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 18, United States Code, to incr...","Tlaib, Rashida [Rep.-D-MI-12]",2026-03-12,House - Judiciary,Referred to the House Committee on the Judiciary.,2026-03-12,6,public_safety,"[G000586, L000602, N000147, O000173, R000617, ..."
4,H.R. 7929,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To direct the Secretary of Transportation to i...,"Titus, Dina [Rep.-D-NV-1]",2026-03-12,"House - Science, Space, and Technology","Referred to the House Committee on Science, Sp...",2026-03-12,0,public_safety,[]


In [11]:
# save to csv
bills_df.to_csv('../../data/bills/bills_with_topic.csv')